## Model #1 Prep - 4x ETFS + Headline and Article Content Sentiment Scores 

In [33]:
import pandas as pd
import json

In [34]:
NEWS_DATA_FOLDER = './data/articles_sentiment_processed/'    # folder with processed sentiment news
qqq = pd.read_csv("./data/raw/QQQ.csv")
spx = pd.read_csv("./data/raw/GSPC.csv")
spy = pd.read_csv("./data/raw/SPY.csv")
es = pd.read_csv("./data/raw/ES=F.csv")

In [35]:
for df in [qqq, spx, spy, es]:
    df["Date"] = pd.to_datetime(df["Date"])

qqq = qqq.rename(columns=lambda x: f"qqq_{x}" if x != "Date" else x)
spx = spx.rename(columns=lambda x: f"spx_{x}" if x != "Date" else x)
spy = spy.rename(columns=lambda x: f"spy_{x}" if x != "Date" else x)
es = es.rename(columns=lambda x: f"es_{x}" if x != "Date" else x)

stock_data = spy.copy()  # Start with SPY
for df in [qqq, spx, es]:
    stock_data = pd.merge(stock_data, df, on="Date", how="inner")  # only keep dates common to all ETFs

# Sort by date just in case
stock_data = stock_data.sort_values("Date")

stock_data = stock_data.rename(columns={"Date": "date"})

# Quick look
print(stock_data.head())

        date  spy_Close   spy_High    spy_Low   spy_Open  spy_Volume  \
0 2010-01-04  85.768440  85.813847  84.391060  85.041910   118944600   
1 2010-01-05  85.995476  86.033318  85.405171  85.715463   111579900   
2 2010-01-06  86.056046  86.267949  85.844142  85.912251   116074400   
3 2010-01-07  86.419289  86.525241  85.654916  85.897093   131091100   
4 2010-01-08  86.706879  86.744721  86.018191  86.192253   126402800   

   qqq_Close   qqq_High    qqq_Low   qqq_Open  ...    spx_Close     spx_High  \
0  40.485809  40.546864  40.354987  40.407318  ...  1132.989990  1133.869995   
1  40.485809  40.555584  40.259048  40.459645  ...  1136.520020  1136.630005   
2  40.241611  40.599198  40.180560  40.468376  ...  1137.140015  1139.189941   
3  40.267796  40.355014  40.049755  40.302683  ...  1141.689941  1142.459961   
4  40.599197  40.599197  40.058457  40.180559  ...  1144.979980  1145.390015   

       spx_Low     spx_Open  spx_Volume  es_Close  es_High   es_Low  es_Open  \
0  111

In [36]:
# Create a "future close" shifted -1 day
# stock_data["spy_Close_future"] = stock_data["spy_Close"].shift(-1)

# Create a binary "direction" label: 1 if up, 0 if down
# stock_data["target"] = (stock_data["spy_Close_future"] > stock_data["spy_Close"]).astype(int)

# Drop the last row because it will have NaN in 'spy_close_future'
# stock_data = stock_data.dropna()
# stock_data = stock_data.drop(columns=["spy_Close_future"])


# Quick look at target distribution
# print(stock_data["target"].value_counts(normalize=True))


In [37]:
import os
import pandas as pd

# Path to your news data directory
news_parts_dir = "./data/processed/merged_sentiment_data.csv"

# Load and concatenate all CSV files
news_df = pd.read_csv(news_parts_dir)

# Convert 'date_published' to datetime, and drop rows with missing essential data
news_df["date_published"] = pd.to_datetime(news_df["date_published"], errors="coerce")
# 🧠 Now average separately:
daily_headline_sentiment = news_df.groupby(news_df["date_published"].dt.date)["headline_sentiment"].mean().reset_index()
daily_article_sentiment = news_df.groupby(news_df["date_published"].dt.date)["article_sentiment"].mean().reset_index()

# Rename for clarity
daily_headline_sentiment.columns = ["date", "avg_headline_sentiment"]
daily_article_sentiment.columns = ["date", "avg_article_sentiment"]

# Merge them together
daily_sentiment = pd.merge(daily_headline_sentiment, daily_article_sentiment, on="date")

# Convert date to datetime
daily_sentiment["date"] = pd.to_datetime(daily_sentiment["date"])

# Preview
print(daily_sentiment.head())

        date  avg_headline_sentiment  avg_article_sentiment
0 2012-05-18           -1.467220e-08               0.809054
1 2012-06-02           -6.502028e-01              -0.663147
2 2012-06-03           -3.310511e-01              -0.367127
3 2012-06-04           -3.935815e-01              -0.389618
4 2012-06-05           -5.879325e-01              -0.444026


In [38]:
stock_data = stock_data[stock_data["date"] >= "2014-06-01"]
daily_sentiment = daily_sentiment[daily_sentiment["date"] >= "2014-06-01"]

merged_df = pd.merge(stock_data, daily_sentiment, on="date", how="left")
merged_df["avg_headline_sentiment"] = merged_df["avg_headline_sentiment"].fillna(0)
merged_df["avg_article_sentiment"] = merged_df["avg_article_sentiment"].fillna(0)
merged_df = merged_df.sort_values("date")
merged_df.head()

,date,spy_Close,spy_High,spy_Low,spy_Open,spy_Volume,qqq_Close,qqq_High,qqq_Low,qqq_Open,...,spx_Low,spx_Open,spx_Volume,es_Close,es_High,es_Low,es_Open,es_Volume,avg_headline_sentiment,avg_article_sentiment
0,2014-06-02,159.182388,159.256666,158.414952,159.223651,64656000,83.556450,83.757968,83.025175,83.739645,...,1915.979980,1923.869995,2509020000,1921.75,1924.25,1913.75,1921.25,1053526,-0.533892,-0.575179
1,2014-06-03,159.099869,159.182382,158.646003,158.794534,65047000,83.519829,83.657228,83.171756,83.281670,...,1918.790039,1923.069946,2867180000,1922.00,1923.50,1916.00,1921.50,948001,-0.511021,-0.618714
2,2014-06-04,159.421646,159.512419,158.662458,158.827496,55529000,83.831261,83.968660,83.180910,83.309148,...,1918.599976,1923.060059,2793920000,1925.75,1927.00,1916.00,1922.75,917108,-0.563256,-0.580557
3,2014-06-05,160.461456,160.626495,159.017344,159.603247,92103000,84.536583,84.692299,83.620594,83.904549,...,1922.930054,1928.520020,3113270000,1938.50,1940.75,1921.00,1925.50,1676659,-0.549064,-0.616599
4,2014-06-06,161.228867,161.270117,160.733738,160.808003,78696000,85.022064,85.022064,84.683146,84.848024,...,1942.410034,1942.410034,2864300000,1949.25,1949.75,1937.75,1938.50,1205739,-0.457761,-0.572454


In [39]:
merged_df.to_csv("./data/processed/1_merged_stock_sentiment.csv", index=False)